### 4.10.1 Démarrage de la session Spark

In [ ]:
# L'exécution de cette cellule démarre l'application Spark

Starting Spark application


SparkSession available as 'spark'.


<u>Affichage des informations sur la session en cours et liens vers Spark UI</u> :

In [ ]:
%%info

### 4.10.2 Installation des packages

Les packages nécessaires ont été installé via l'étape de **bootstrap** à l'instanciation du serveur.

### 4.10.3 Import des librairies

In [ ]:
import pandas as pd
import numpy as np
import io
import os
import tensorflow as tf
from tensorflow import keras
from PIL import Image
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
import matplotlib.pyplot as plt

### 4.10.4 Définition des PATH pour charger les images et enregistrer les résultats

Nous accédons directement à nos **données sur S3** comme si elles étaient **stockées localement**.

In [ ]:
PATH = 's3://my-bucket'
PATH_Data = PATH+'/Test'
PATH_Result = PATH+'/Results'
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

PATH:        s3://my-bucket
PATH_Data:   s3://my-bucket/Test
PATH_Result: s3://my-bucket/Results

### 4.10.5 Traitement des données

#### 4.10.5.1 Chargement des données

In [ ]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

In [ ]:
images.show(5)

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|s3://my-bucket...|2023-04-26 16:30:23|125135|[FF D8 FF E0 00 1...|
|s3://my-bucket...|2023-04-26 16:30:23|124785|[FF D8 FF E0 00 1...|
|s3://my-bucket...|2023-04-26 16:30:23|123514|[FF D8 FF E0 00 1...|
|s3://my-bucket...|2023-04-26 16:30:23|122958|[FF D8 FF E0 00 1...|
|s3://my-bucket...|2023-04-26 16:30:23|122807|[FF D8 FF E0 00 1...|
+--------------------+-------------------+------+--------------------+
only showing top 5 rows

<u>Je ne conserve que le **path** de l'image et j'ajoute <br />
    une colonne contenant les **labels** de chaque image</u> :

In [ ]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None
+---------------------------------------------+-----------+
|path                                         |label      |
+---------------------------------------------+-----------+
|s3://my-bucket/Test/apple_hit_1/r0_115.jpg|apple_hit_1|
|s3://my-bucket/Test/apple_hit_1/r0_119.jpg|apple_hit_1|
|s3://my-bucket/Test/apple_hit_1/r0_107.jpg|apple_hit_1|
|s3://my-bucket/Test/apple_hit_1/r0_143.jpg|apple_hit_1|
|s3://my-bucket/Test/apple_hit_1/r0_111.jpg|apple_hit_1|
+---------------------------------------------+-----------+
only showing top 5 rows

None

#### 4.10.5.2 Préparation du modèle

In [ ]:
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

In [ ]:
new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

In [ ]:
brodcast_weights = sc.broadcast(new_model.get_weights())

In [ ]:
new_model.summary()

In [ ]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(brodcast_weights.value)
    return new_model

#### 4.10.5.3 Définition du processus de chargement des images <br/> et application de leur featurisation à travers l'utilisation de pandas UDF

In [ ]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224]) #Open image from binary objects content and resize it
    arr = img_to_array(img) # Convert img to to array
    return preprocess_input(arr) #Reshape to (224,224,3)

def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn() 
    #Weights broadcasting
    model.set_weights(brodcast_weights.value)
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

/mnt1/yarn/usercache/livy/appcache/application_1683515387863_0001/container_1683515387863_0001_01_000001/pyspark.zip/pyspark/sql/pandas/functions.py:398: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

#### 4.10.5.4 Exécutions des actions d'extractions de features

In [ ]:
# spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")

In [ ]:
features_df = images.repartition(24).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

In [ ]:
print(PATH_Result)

s3://my-bucket/Results

In [ ]:
features_df.write.mode("overwrite").parquet(PATH_Result)

### 4.10.6 Chargement des données enregistrées et validation du résultat

In [ ]:
df = pd.read_parquet(PATH_Result, engine='pyarrow')

In [ ]:
df.head()

                                            path  ...                                           features
0  s3://my-bucket/Test/apple_hit_1/r1_299.jpg  ...  [0.88818276, 0.9766362, 1.6814375, 0.0, 0.7473...
1  s3://my-bucket/Test/apple_hit_1/r0_167.jpg  ...  [0.23584183, 0.13031594, 1.4356456, 0.0, 0.357...
2   s3://my-bucket/Test/apple_hit_1/r1_43.jpg  ...  [0.008909265, 0.20073241, 2.0521822, 0.0, 0.17...
3  s3://my-bucket/Test/apple_hit_1/r2_203.jpg  ...  [0.07983717, 0.30655688, 0.12255844, 0.0, 0.51...
4   s3://my-bucket/Test/apple_hit_1/r1_63.jpg  ...  [0.16827433, 0.17030035, 0.27944311, 0.0, 0.48...

[5 rows x 3 columns]

In [ ]:
df.loc[0,'features'].shape

(1280,)

In [ ]:
df.shape

(3110, 3)

In [ ]:
df1 = spark.read.parquet(PATH_Result)

In [ ]:
df1.printSchema()

root
 |-- path: string (nullable = true)
 |-- label: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)

## 4.11.6 Standard scalering image features

In [ ]:
from pyspark.ml.linalg import VectorUDT, Vectors
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, DoubleType

# define a UDF for converting numpy arrays to PySpark VectorUDT objects
numpy_to_vector = udf(lambda a: Vectors.dense(a), returnType=VectorUDT())

# apply the UDF to the 'features' column to create a new column of VectorUDT objects
df1 = df1.withColumn('features_vector', numpy_to_vector(df1['features']))
df1 = df1.withColumn('label_red', element_at(split(df1['label'], '_'),1))
# drop the original 'features' column
#df = df.drop('features')

In [ ]:
df1.show()

In [ ]:
scaler = StandardScaler(
    inputCol = 'features_vector', 
    outputCol = 'scaledFeatures',
    withMean = True,
    withStd = True
).fit(df1)

# when we transform the dataframe, the old
# feature will still remain in it
df_scaled = scaler.transform(df1)
df_scaled.show(6)

+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|                path|      label|            features|     features_vector|label_red|      scaledFeatures|
+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|s3://my-bucket...|apple_hit_1|[0.3821219, 0.231...|[0.38212189078330...|    apple|[0.04212617474026...|
|s3://my-bucket...|apple_hit_1|[0.7551134, 1.278...|[0.75511342287063...|    apple|[0.91827456645018...|
|s3://my-bucket...|apple_hit_1|[0.77546775, 1.28...|[0.77546775341033...|    apple|[0.96608641318808...|
|s3://my-bucket...|apple_hit_1|[0.3894314, 0.012...|[0.38943138718605...|    apple|[0.05929601096048...|
|s3://my-bucket...|apple_hit_1|[0.0, 0.2023897, ...|[0.0,0.2023897022...|    apple|[-0.8554692161614...|
|s3://my-bucket...|apple_hit_1|[0.60624766, 0.01...|[0.60624766349792...|    apple|[0.56859237681661...|
+--------------------+-----------+------------

In [ ]:
df_scaled.printSchema()

root
 |-- path: string (nullable = true)
 |-- label: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- features_vector: vector (nullable = true)
 |-- label_red: string (nullable = true)
 |-- scaledFeatures: vector (nullable = true)

## 3.10 Performing PCA

In [ ]:
n_components = 20
pca = PCA(
    k = n_components, 
    inputCol = 'scaledFeatures', 
    outputCol = 'pcaFeatures'
).fit(df_scaled)
print('Cumulative variance', pca.explainedVariance.toArray().sum())

Cumulative variance 0.5376154532383266

In [ ]:
variances = pca.explainedVariance.toArray()
# Compute the cumulative explained variance
cumulative_variances = np.cumsum(variances)

# Plot the explained variance ratios
plt.bar(range(1, n_components + 1), variances, label='Individual explained variance')
plt.plot(range(1, n_components + 1), cumulative_variances,'-s', label='Cumulative explained variance', color='red')
plt.xlabel('Number of principal components')
plt.ylabel('Explained variance ratio')
plt.legend(loc='best')
plt.show()

In [ ]:
n_components = 2
Pca = PCA(
    k = n_components, 
    inputCol = 'scaledFeatures', 
    outputCol = 'pcaFeatures'
).fit(df_scaled)

df_pca = Pca.transform(df_scaled)
print('Explained Variance Ratio', Pca.explainedVariance.toArray())
df_pca.show(6)
X_pca = df_pca.rdd.map(lambda row: row.pcaFeatures).collect()
X_pca = np.array(X_pca)

Explained Variance Ratio [0.1133648  0.06571123]
+--------------------+-----------+--------------------+--------------------+---------+--------------------+--------------------+
|                path|      label|            features|     features_vector|label_red|      scaledFeatures|         pcaFeatures|
+--------------------+-----------+--------------------+--------------------+---------+--------------------+--------------------+
|s3://my-bucket...|apple_hit_1|[0.3821219, 0.231...|[0.38212189078330...|    apple|[0.04212617474026...|[1.56348457055567...|
|s3://my-bucket...|apple_hit_1|[0.7551134, 1.278...|[0.75511342287063...|    apple|[0.91827456645018...|[12.0309400335016...|
|s3://my-bucket...|apple_hit_1|[0.77546775, 1.28...|[0.77546775341033...|    apple|[0.96608641318808...|[7.84802238865728...|
|s3://my-bucket...|apple_hit_1|[0.3894314, 0.012...|[0.38943138718605...|    apple|[0.05929601096048...|[-1.4119891840638...|
|s3://my-bucket...|apple_hit_1|[0.0, 0.2023897, ...|[0.0,0.2

In [ ]:
# change default style figure and font size
def plot_pca(X, y):
    """Plot of the 2-dimensional image features"""
    labels = np.array(df_pca.select('label').rdd.flatMap(lambda x: x).collect())
    u_labels = np.unique(labels)

    for i in u_labels:
        plt.scatter(X[labels == i , 0] , X[labels == i , 1] , label = f'{i}')
    plt.title(f'PCA Scatter plot', fontsize=14 )
    plt.legend(loc='upper right')# bbox_to_anchor=(0.5, -0.2), ncol=len(u_labels)) #,
    plt.show()

In [ ]:
plot_pca(X_pca, "label_red")

Save PCA dataframe

In [ ]:
PATH_pca = PATH+'/Results/PcaFeatures'
df_pca.write.mode("overwrite").parquet(PATH_pca)